# Brain Tumor DX — Colab GPU Training

This notebook trains the classifier and/or segmentation model on Colab's free **T4 GPU**.

**Steps before running:**
1. Upload your dataset to Colab (cell 2)
2. Upload the project code (cell 3) or clone from GitHub
3. Run training cells
4. Download trained checkpoints (final cell)

In [ ]:
# Cell 1: Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', None))
    if vram:
        print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU found! Go to Runtime > Change runtime type > select GPU."
    )

In [ ]:
# Cell 2: Upload your dataset
# Option A: Upload from your local machine (run this, then pick your data folder)
import os
from google.colab import files
import zipfile

os.makedirs("/content/data", exist_ok=True)

print("Upload your dataset as a .zip file (classification or segmentation data):")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith(".zip"):
        print(f"Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall("/content/data")
        print(f"Extracted to /content/data")
    else:
        os.rename(filename, f"/content/data/{filename}")

print("\nContents of /content/data:")
!ls -la /content/data/

In [ ]:
# Cell 3: Clone or upload project code
# Option A: Clone from GitHub (if repo is pushed)
# !git clone https://github.com/YOUR_USERNAME/brain-tumor-dx.git /content/brain-tumor-dx

# Option B: Upload the project zip (run from your local terminal first):
#   cd D:\brain-tumor-dx\brain-tumor-dx
#   tar -czf brain-tumor-dx.tar.gz --exclude=.venv --exclude=__pycache__ --exclude=.git .
# Then upload brain-tumor-dx.tar.gz in the next cell.

# Option C: Mount Google Drive (if you synced the project there)
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to where your project lives on Drive
PROJECT_DIR = "/content/drive/MyDrive/brain-tumor-dx"
!ls {PROJECT_DIR}

In [ ]:
# Cell 4: Install dependencies
!pip install -q torch torchvision monai nibabel pydicom scikit-image tqdm
!pip install -q pydantic python-dotenv

# Verify torch sees the GPU
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# Cell 5: Train the CLASSIFIER (ResNet50)
# Adjust paths to match where you uploaded data
import sys
sys.path.insert(0, f"{PROJECT_DIR}/src")

os.environ["DEVICE"] = "cuda"

!cd {PROJECT_DIR} && python -m brain_tumor_dx.training.train_classifier \
    --data-root /content/data/classification \
    --epochs 15 \
    --batch-size 32 \
    --lr 1e-4 \
    --out checkpoints/classifier.pt

In [ ]:
# Cell 5 alt: Train classifier INLINE (if module import doesn't work)
import sys, os
os.environ["DEVICE"] = "cuda"
sys.path.insert(0, f"{PROJECT_DIR}/src")

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

# -- Import project modules --
from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import ClassificationDataset
from brain_tumor_dx.models.classifier import TumorClassifier

DATA_ROOT = "/content/data/classification"  # <-- adjust this
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-4
OUT = "checkpoints/classifier.pt"

os.makedirs("checkpoints", exist_ok=True)

dataset = ClassificationDataset(DATA_ROOT)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

model = TumorClassifier(num_classes=len(settings.tumor_classes)).to("cuda")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc=f"epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to("cuda"), labels.to("cuda")
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg = running_loss / len(loader)
    print(f"epoch {epoch+1}: avg loss = {avg:.4f}")

torch.save(model.state_dict(), OUT)
print(f"Saved -> {OUT}")

In [ ]:
# Cell 6: Train the SEGMENTER (3D U-Net)
# Adjust paths to match where you uploaded data

!cd {PROJECT_DIR} && python -m brain_tumor_dx.training.train_segmentation \
    --data-root /content/data/segmentation \
    --epochs 50 \
    --batch-size 2 \
    --lr 1e-4 \
    --out checkpoints/segmentation.pt

In [ ]:
# Cell 6 alt: Train segmenter INLINE
import sys, os
os.environ["DEVICE"] = "cuda"
sys.path.insert(0, f"{PROJECT_DIR}/src")

import torch
from monai.losses import DiceLoss
from torch.utils.data import DataLoader
from tqdm import tqdm

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import SegmentationDataset
from brain_tumor_dx.models.segmentation import TumorSegmenter

DATA_ROOT = "/content/data/segmentation"  # <-- adjust this
EPOCHS = 50
BATCH_SIZE = 2
LR = 1e-4
OUT = "checkpoints/segmentation.pt"

os.makedirs("checkpoints", exist_ok=True)

dataset = SegmentationDataset(DATA_ROOT)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

model = TumorSegmenter().to("cuda")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = DiceLoss(sigmoid=True)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(loader, desc=f"epoch {epoch+1}/{EPOCHS}"):
        images, masks = images.to("cuda"), masks.to("cuda")
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg = running_loss / len(loader)
    print(f"epoch {epoch+1}: avg dice loss = {avg:.4f}")

torch.save(model.state_dict(), OUT)
print(f"Saved -> {OUT}")

In [ ]:
# Cell 7: Download trained checkpoints
from google.colab import files

for ckpt in ["checkpoints/classifier.pt", "checkpoints/segmentation.pt"]:
    if os.path.exists(ckpt):
        print(f"Downloading {ckpt}...")
        files.download(ckpt)
    else:
        print(f"{ckpt} not found — skipping")

In [ ]:
# Cell 8: (Optional) Save checkpoints to Google Drive
import shutil

DRIVE_CKPT_DIR = "/content/drive/MyDrive/brain-tumor-dx-checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

for ckpt in ["checkpoints/classifier.pt", "checkpoints/segmentation.pt"]:
    if os.path.exists(ckpt):
        shutil.copy(ckpt, DRIVE_CKPT_DIR)
        print(f"Copied {ckpt} -> {DRIVE_CKPT_DIR}")

print("Done. Checkpoints saved to Google Drive.")